# Reward Scoring Notebook

This notebook scores two summary datasets with the project reward function:

1. GPT-4o teacher summaries
2. T5-small baseline predictions

The reward function combines:
- tweet-dominant relevance
- MiniCheck factuality
- role coverage
- urgency coverage

The goal is to produce reward JSONL files and presentation/analysis CSV reports for comparison.

## Quick Kaggle Environment Check:

In [1]:
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", os.getcwd())
print("Kaggle working exists:", Path("/kaggle/working").exists())
print("Kaggle input exists:", Path("/kaggle/input").exists())

print("\nGPU check:")
!nvidia-smi

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Current working directory: /kaggle/working
Kaggle working exists: True
Kaggle input exists: True

GPU check:
Mon Jun 29 05:37:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8

## Clone The Project Repository

This cell clones the GitHub repository into `/kaggle/working`, switches into the project directory, and verifies that the reward scorer, generated GPT-4o summaries, and modeling folders are available.

The notebook should use the latest pushed version of the reward scorer, including the updated role coverage and urgency scoring logic.

In [2]:
from pathlib import Path
import os

REPO_URL = "https://github.com/ffaustin17/role-aware-crisis-summarization.git"
REPO_DIR = Path("/kaggle/working/role-aware-crisis-summarization")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists. Pulling latest changes...")
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}

print("\nLatest commit:")
!git log --oneline --decorate -5

print("\nRepository status:")
!git status --short --branch

print("\nKey scripts:")
!ls -lh scripts/score_rewards.py scripts/analyze_summary_reward_dataset.py scripts/build_prediction_reward_report.py

print("\nGenerated summary files:")
!ls -lh data/generated || true

print("\nModeling data folders:")
!find data/modeling -maxdepth 3 -type f -name "*.jsonl" -print | sort || true

print("\nReward files currently in repo:")
!find data/rewards -maxdepth 2 -type f -print | sort || true

Cloning into '/kaggle/working/role-aware-crisis-summarization'...
remote: Enumerating objects: 300, done.
remote: Counting objects: 100% (300/300), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 300 (delta 150), reused 226 (delta 79), pack-reused 0 (from 0)
Receiving objects: 100% (300/300), 9.68 MiB | 15.05 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/kaggle/working/role-aware-crisis-summarization

Latest commit:
e050d38 (HEAD -> main, origin/main, origin/HEAD) Add T5 baseline v2 modeling outputs
58b06f1 Generalize reward reporting scripts
59d48ab Tighten reward role and urgency scoring
9939c14 Move notebooks into notebooks folder
442aefe Kaggle Notebook | t5-training-and-eval-notebook | Version 1

Repository status:
## main...origin/main

Key scripts:
-rw-r--r-- 1 root root  13K Jun 29 05:37 scripts/analyze_summary_reward_dataset.py
-rw-r--r-- 1 root root 9.8K Jun 29 05:37 scripts/build_prediction_reward_report.py
-rw-r--r-- 1 root root  35K Jun 29 05

## Install And Verify Reward Scoring Dependencies

The reward scorer needs SentenceTransformer for relevance and MiniCheck for factuality. Kaggle may print dependency resolver warnings because its base image includes many GPU/RAPIDS packages. The important check is whether the required packages import successfully afterward.

In [3]:
%cd /kaggle/working/role-aware-crisis-summarization

!pip install -q sentence-transformers
!pip install -q git+https://github.com/Liyan06/MiniCheck.git

/kaggle/working/role-aware-crisis-summarization
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0

## Dependency Import Check

In [4]:
import torch
import sentence_transformers

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
print("sentence_transformers:", sentence_transformers.__version__)

try:
    from minicheck.minicheck import MiniCheck
    print("MiniCheck import: OK")
except Exception as exc:
    print("MiniCheck import failed:", repr(exc))
    raise

torch: 2.10.0+cu128
cuda available: True
gpu count: 2
sentence_transformers: 5.4.0
MiniCheck import: OK


## Smoke Test The Reward Scorer

Before scoring all 6001 summaries, this cell runs a tiny MiniCheck-based scoring job on both datasets:

1. T5 baseline v2 predictions
2. GPT-4o teacher summaries

This confirms that the scorer can read both input schemas and produce reward JSONL outputs with the updated relevance, role coverage, urgency, and MiniCheck factuality fields.

In [5]:
%cd /kaggle/working/role-aware-crisis-summarization

!mkdir -p temp

print("Scoring 3 T5 baseline v2 predictions...")
!python scripts/score_rewards.py \
  --input data/modeling/t5_baseline_v2/all_predictions.jsonl \
  --output temp/t5_v2_reward_smoke.jsonl \
  --summary-csv temp/t5_v2_reward_smoke.csv \
  --summary-md temp/t5_v2_reward_smoke.md \
  --relevance-backend sentence-transformer \
  --factuality-backend minicheck \
  --max-records 3

print("\nScoring 3 GPT-4o teacher summaries...")
!python scripts/score_rewards.py \
  --input data/generated/gpt4o_initial_summaries_v0203.jsonl \
  --output temp/gpt4o_teacher_reward_smoke.jsonl \
  --summary-csv temp/gpt4o_teacher_reward_smoke.csv \
  --summary-md temp/gpt4o_teacher_reward_smoke.md \
  --relevance-backend sentence-transformer \
  --factuality-backend minicheck \
  --max-records 3

/kaggle/working/role-aware-crisis-summarization
Scoring 3 T5 baseline v2 predictions...
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 584kB/s]
README.md: 10.5kB [00:00, 35.8MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 402kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 4.68MB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:02<00:00, 40.6MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1455.94it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.39MB/s]
vocab.txt: 232kB [00:00, 11.0MB/s]
tokenizer.json: 466kB [00:00, 39.6MB/s

## Inspect Smoke Outputs

In [6]:
import json
from pathlib import Path

for path in [
    Path("temp/t5_v2_reward_smoke.jsonl"),
    Path("temp/gpt4o_teacher_reward_smoke.jsonl"),
]:
    print("\n==", path, "==")
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("records:", len(records))
    first = records[0]
    print("tweet_id:", first["tweet_id"])
    print("role:", first["role"])
    print("prediction_text:", first["prediction_text"])
    print("reward:", first["reward"])
    print("component_scores:", first["component_scores"])
    print("relevance_details keys:", sorted(first["relevance_details"].keys()))
    print("role_coverage_details keys:", sorted(first["role_coverage_details"].keys()))
    print("urgency_details keys:", sorted(first["urgency_details"].keys()))
    print("factuality backend:", first["factuality_details"].get("backend"))
    print("sentence scores:", first["factuality_details"].get("sentence_scores", [])[:1])


== temp/t5_v2_reward_smoke.jsonl ==
records: 3
tweet_id: 1
role: Firefighter
prediction_text: Firefighters should assess fire hazards and ensure safety in areas affected by the wildfire.
reward: 0.6768481841683388
component_scores: {'relevance': 0.7327283084392547, 'factuality': 0.48157310485839844, 'role_coverage': 0.5, 'urgency': 1.0}
relevance_details keys: ['context_relevance', 'context_source_available', 'context_weight', 'fallback_source', 'tweet_relevance', 'tweet_source_available', 'tweet_weight']
role_coverage_details keys: ['applicable_category_count', 'covered_category_count', 'roles', 'score_reason']
urgency_details keys: ['candidate_categories', 'candidate_terms_by_category', 'covered_categories', 'score_reason', 'source_categories', 'source_terms_by_category']
factuality backend: minicheck
sentence scores: [{'claim': 'Firefighters should assess fire hazards and ensure safety in areas affected by the wildfire.', 'predicted_label': 0, 'support_probability': 0.4815731048583

## Full Reward Scoring: T5 Baseline V2 Predictions

This cell scores all 6001 T5-small baseline v2 predictions with the project reward function.

Output artifacts:
- reward JSONL
- compact reward summary CSV
- compact reward summary Markdown

The JSONL file is the main artifact. The summary CSV/Markdown gives quick aggregate means by overall group and role label.

In [7]:
%cd /kaggle/working/role-aware-crisis-summarization

!mkdir -p data/rewards reports/tables

!python scripts/score_rewards.py \
  --input data/modeling/t5_baseline_v2/all_predictions.jsonl \
  --output data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl \
  --summary-csv reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.csv \
  --summary-md reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.md \
  --relevance-backend sentence-transformer \
  --factuality-backend minicheck

/kaggle/working/role-aware-crisis-summarization
Loading weights: 100%|█| 103/103 [00:00<00:00, 1564.70it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█| 560/560 [00:00<00:00, 595.26it/s, Materializing param=s
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_w

## Validate T5 Reward Output

In [8]:
from pathlib import Path
import json
from collections import Counter

path = Path("data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl")
records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

print("records:", len(records))
print("tweet_id unique:", len({record["tweet_id"] for record in records}))
print("role counts:", dict(Counter(record["role"] for record in records)))
print("first reward:", records[0]["reward"])
print("first component scores:", records[0]["component_scores"])
print("last tweet_id:", records[-1]["tweet_id"])

print("\nSummary markdown:")
print(Path("reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.md").read_text(encoding="utf-8"))

records: 6001
tweet_id unique: 6001
role counts: {'Firefighter': 1023, 'Police': 3280, 'Police/Firefighter': 340, 'EMS': 593, 'Police/EMS': 401, 'Firefighter/EMS': 153, 'Police/Firefighter/EMS': 211}
first reward: 0.6768481951206923
first component scores: {'relevance': 0.7327283397316933, 'factuality': 0.48157310485839844, 'role_coverage': 0.5, 'urgency': 1.0}
last tweet_id: 6001

Summary markdown:
| group | num_examples | reward_mean | relevance_mean | factuality_mean | role_coverage_mean | urgency_mean |
|---|---:|---:|---:|---:|---:|---:|
| overall | 6001 | 0.593620 | 0.719472 | 0.480305 | 0.471233 | 0.637408 |
| EMS | 593 | 0.639708 | 0.757233 | 0.480672 | 0.630551 | 0.641990 |
| Firefighter | 1023 | 0.669614 | 0.749768 | 0.481567 | 0.592538 | 0.841479 |
| Firefighter/EMS | 153 | 0.635047 | 0.692068 | 0.481531 | 0.637800 | 0.724401 |
| Police | 3280 | 0.586761 | 0.715906 | 0.480105 | 0.459959 | 0.620879 |
| Police/EMS | 401 | 0.531742 | 0.675635 | 0.477736 | 0.401704 | 0.477473 |


## Full Reward Scoring: GPT-4o Teacher Summaries

This cell scores all 6001 GPT-4o teacher summaries with the same reward function used for the T5 baseline predictions.

This gives us a direct comparison point:
- GPT-4o prompted teacher summary reward distribution
- T5-small baseline prediction reward distribution

Both are scored against the same source tweet/context using the same reward criteria.

In [9]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/score_rewards.py \
  --input data/generated/gpt4o_initial_summaries_v0203.jsonl \
  --output data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl \
  --summary-csv reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.csv \
  --summary-md reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.md \
  --relevance-backend sentence-transformer \
  --factuality-backend minicheck

/kaggle/working/role-aware-crisis-summarization
Loading weights: 100%|█| 103/103 [00:00<00:00, 1547.55it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█| 560/560 [00:00<00:00, 596.41it/s, Materializing param=s
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_w

Validate GPT-4o Reward Output

In [10]:
from pathlib import Path
import json
from collections import Counter

path = Path("data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl")
records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

print("records:", len(records))
print("tweet_id unique:", len({record["tweet_id"] for record in records}))
print("role counts:", dict(Counter(record["role"] for record in records)))
print("first reward:", records[0]["reward"])
print("first component scores:", records[0]["component_scores"])
print("last tweet_id:", records[-1]["tweet_id"])

print("\nSummary markdown:")
print(Path("reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.md").read_text(encoding="utf-8"))

records: 6001
tweet_id unique: 6001
role counts: {'Firefighter': 1023, 'Police': 3280, 'Police/Firefighter': 340, 'EMS': 593, 'Police/EMS': 401, 'Firefighter/EMS': 153, 'Police/Firefighter/EMS': 211}
first reward: 0.7956234510987997
first component scores: {'relevance': 0.7904398813843727, 'factuality': 0.475877970457077, 'role_coverage': 1.0, 'urgency': 1.0}
last tweet_id: 5860

Summary markdown:
| group | num_examples | reward_mean | relevance_mean | factuality_mean | role_coverage_mean | urgency_mean |
|---|---:|---:|---:|---:|---:|---:|
| overall | 6001 | 0.617690 | 0.724957 | 0.478697 | 0.508129 | 0.713276 |
| EMS | 593 | 0.729448 | 0.753576 | 0.476748 | 0.909500 | 0.823047 |
| Firefighter | 1023 | 0.726688 | 0.741738 | 0.481340 | 0.865266 | 0.868459 |
| Firefighter/EMS | 153 | 0.610175 | 0.668411 | 0.475728 | 0.703704 | 0.582789 |
| Police | 3280 | 0.574946 | 0.730602 | 0.479054 | 0.314685 | 0.682673 |
| Police/EMS | 401 | 0.575733 | 0.680017 | 0.476230 | 0.466334 | 0.627016 |
| 

## Generate Reward Distribution Analysis Tables

This cell creates descriptive analysis tables for both scored datasets.

For each dataset, the analyzer writes:
- overall metrics
- metrics by exact role label
- metrics by disaster type
- metrics by information type
- histogram-style distributions

These reports help compare reward behavior across model outputs and dataset slices.

In [11]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/analyze_summary_reward_dataset.py \
  --input data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl \
  --output-dir reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck

!python scripts/analyze_summary_reward_dataset.py \
  --input data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl \
  --output-dir reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck

print("\nT5 analysis files:")
!find reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck -type f -maxdepth 1 -print | sort

print("\nGPT-4o analysis files:")
!find reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck -type f -maxdepth 1 -print | sort

/kaggle/working/role-aware-crisis-summarization
Analyzed records: 6001
Wrote analysis CSVs to: reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck
Analyzed records: 6001
Wrote analysis CSVs to: reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck

T5 analysis files:
find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/by_disaster_type.csv
reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/by_information_type.csv
reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/by_role_label.csv
reports/tables/summary_rewa

## Preview of overall analysis

In [12]:
from pathlib import Path
import pandas as pd

paths = {
    "t5_v2": Path("reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/overall_metrics.csv"),
    "gpt4o": Path("reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck/overall_metrics.csv"),
}

for name, path in paths.items():
    print("\n==", name, "==")
    display(pd.read_csv(path))


== t5_v2 ==


,group_field,group_value,numeric_field,count,mean,median,std,min,p10,p25,p50,p75,p90,max
0,overall,overall,reward,6001,0.593620,0.585116,0.125431,0.323963,0.432081,0.475538,0.585116,0.691915,0.769603,0.828158
1,overall,overall,relevance,6001,0.719472,0.720343,0.053101,0.556790,0.651270,0.685626,0.720343,0.753878,0.788289,0.884343
2,overall,overall,factuality,6001,0.480305,0.477637,0.009620,0.464718,0.471211,0.473540,0.477637,0.484166,0.493997,0.527331
3,overall,overall,role_coverage,6001,0.471233,0.500000,0.433378,0.000000,0.000000,0.000000,0.500000,1.000000,1.000000,1.000000
4,overall,overall,urgency,6001,0.637408,0.500000,0.361165,0.000000,0.000000,0.400000,0.500000,1.000000,1.000000,1.000000
5,overall,overall,summary_word_count,6001,16.719213,16.000000,2.603391,7.000000,14.000000,15.000000,16.000000,18.000000,20.000000,34.000000
6,overall,overall,summary_char_length,6001,117.880187,116.000000,18.253022,49.000000,98.000000,106.000000,116.000000,126.000000,141.000000,213.000000
7,overall,overall,summary_sentence_count,6001,1.011998,1.000000,0.111905,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000



== gpt4o ==


,group_field,group_value,numeric_field,count,mean,median,std,min,p10,p25,p50,p75,p90,max
0,overall,overall,reward,6001,0.617690,0.601125,0.124723,0.326106,0.454620,0.515992,0.601125,0.743167,0.786183,0.832580
1,overall,overall,relevance,6001,0.724957,0.729357,0.053980,0.569136,0.649571,0.688499,0.729357,0.764637,0.792488,0.883485
2,overall,overall,factuality,6001,0.478697,0.475878,0.009328,0.463212,0.470336,0.472634,0.475878,0.481387,0.491299,0.523977
3,overall,overall,role_coverage,6001,0.508129,0.500000,0.445587,0.000000,0.000000,0.000000,0.500000,1.000000,1.000000,1.000000
4,overall,overall,urgency,6001,0.713276,1.000000,0.343154,0.000000,0.400000,0.400000,1.000000,1.000000,1.000000,1.000000
5,overall,overall,summary_word_count,6001,17.388102,17.000000,2.166607,11.000000,15.000000,16.000000,17.000000,19.000000,20.000000,28.000000
6,overall,overall,summary_char_length,6001,121.824363,122.000000,14.157359,77.000000,104.000000,112.000000,122.000000,131.000000,140.000000,190.000000
7,overall,overall,summary_sentence_count,6001,1.007332,1.000000,0.090992,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000


## Build Presentation CSV Reports

This cell joins each summary dataset with its reward scores into presentation-ready CSV files.

Each CSV foregrounds:
- original tweet
- input text
- candidate summary
- reference summary, when available
- reward and component scores
- role/disaster/information metadata
- source identifiers

The T5 report includes the model prediction as the candidate summary and the GPT-4o teacher summary as the reference. The GPT-4o report uses the teacher summary as the candidate summary and has no separate reference summary.

In [13]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/build_prediction_reward_report.py \
  --summaries data/modeling/t5_baseline_v2/all_predictions.jsonl \
  --rewards data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl \
  --output reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv \
  --expected-count 6001 \
  --expected-first-tweet-id 1

!python scripts/build_prediction_reward_report.py \
  --summaries data/generated/gpt4o_initial_summaries_v0203.jsonl \
  --rewards data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl \
  --output reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv \
  --expected-count 6001 \
  --expected-first-tweet-id 1

/kaggle/working/role-aware-crisis-summarization
Summary records: 6001
Reward records: 6001
Wrote report rows: 6001
Wrote report CSV to: reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv
Summary records: 6001
Reward records: 6001
Wrote report rows: 6001
Wrote report CSV to: reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv


## Validate and Preview Presentation CSVs

In [14]:
import pandas as pd
from pathlib import Path

report_paths = {
    "t5_v2": Path("reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv"),
    "gpt4o": Path("reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv"),
}

for name, path in report_paths.items():
    df = pd.read_csv(path)
    print("\n==", name, "==")
    print("path:", path)
    print("rows:", len(df))
    print("columns:", list(df.columns))
    display(df.head(3))


== t5_v2 ==
path: reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv
rows: 6001
columns: ['original_tweet', 'input_text', 'candidate_summary', 'reference_summary', 'reward', 'relevance', 'tweet_relevance', 'context_relevance', 'factuality', 'minicheck_support_probability', 'minicheck_predicted_label', 'role_coverage', 'urgency', 'role', 'disaster_type', 'information_type', 'tweet_id', 'source_row_id']


,original_tweet,input_text,candidate_summary,reference_summary,reward,relevance,tweet_relevance,context_relevance,factuality,minicheck_support_probability,minicheck_predicted_label,role_coverage,urgency,role,disaster_type,information_type,tweet_id,source_row_id
0,"Yikes, that smoke is too close for comfort. St...",Responder Roles: Firefighter\nDisaster Type: W...,Firefighters should assess fire hazards and en...,Firefighters should assess smoke proximity and...,0.676848,0.732728,0.690430,0.831425,0.481573,0.48157310485839844,0,0.5,1.0,Firefighter,Wildfire,Affected individuals,1,495
1,Colorado floods threaten more homes as another...,Responder Roles: Police\nDisaster Type: Floods...,Police should assess potential criminal activi...,Police should assess potential evacuation need...,0.377086,0.740732,0.715515,0.799569,0.471322,0.47132164239883423,0,0.0,0.0,Police,Floods,Affected individuals,2,13091
2,"RT @mikesbloggity: Until Sunday, Tide is washi...",Responder Roles: Police\nDisaster Type: Floods...,Police should assess crowd control needs and c...,Police should monitor crowd activity at the la...,0.657079,0.683837,0.666347,0.724645,0.470945,0.4709448218345642,0,1.0,0.5,Police,Floods,Donations and volunteering,3,8537



== gpt4o ==
path: reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv
rows: 6001
columns: ['original_tweet', 'input_text', 'candidate_summary', 'reference_summary', 'reward', 'relevance', 'tweet_relevance', 'context_relevance', 'factuality', 'minicheck_support_probability', 'minicheck_predicted_label', 'role_coverage', 'urgency', 'role', 'disaster_type', 'information_type', 'tweet_id', 'source_row_id']


,original_tweet,input_text,candidate_summary,reference_summary,reward,relevance,tweet_relevance,context_relevance,factuality,minicheck_support_probability,minicheck_predicted_label,role_coverage,urgency,role,disaster_type,information_type,tweet_id,source_row_id
0,"Yikes, that smoke is too close for comfort. St...",Responder Roles: Firefighter\nDisaster Type: W...,Firefighters should assess smoke proximity and...,NaN,0.795623,0.790440,0.816154,0.730441,0.475878,0.475877970457077,0,1.0,1.0,Firefighter,Wildfire,Affected individuals,1,495
1,Colorado floods threaten more homes as another...,Responder Roles: Police\nDisaster Type: Floods...,Police should assess potential evacuation need...,NaN,0.386470,0.762714,0.762349,0.763566,0.478079,0.47807931900024414,0,0.0,0.0,Police,Floods,Affected individuals,2,13091
2,"RT @mikesbloggity: Until Sunday, Tide is washi...",Responder Roles: Police\nDisaster Type: Floods...,Police should monitor crowd activity at the la...,NaN,0.444055,0.703312,0.684423,0.747386,0.471585,0.47158530354499817,0,0.0,0.4,Police,Floods,Donations and volunteering,3,8537


## Package Reward Scoring Artifacts

This cell lists the generated reward artifacts and packages the important outputs into a single zip file for download.

The zip includes:
- reward JSONL files for T5 v2 predictions and GPT-4o teacher summaries
- compact reward summary CSV/Markdown files
- detailed reward analysis CSV directories
- presentation-ready CSV reports

In [15]:
%cd /kaggle/working/role-aware-crisis-summarization

print("Reward JSONL files:")
!ls -lh data/rewards/*v2* data/rewards/*gpt4o* || true

print("\nReward summary tables:")
!ls -lh reports/tables/*reward_summary*tweet_relevance_minicheck* || true

print("\nPresentation reports:")
!ls -lh reports/tables/*with_rewards_report.csv || true

print("\nDetailed analysis directories:")
!find reports/tables/summary_reward_analysis \
  -maxdepth 2 \
  -type f \
  \( -path "*t5_baseline_v2_all_predictions_tweet_relevance_minicheck*" -o -path "*gpt4o_initial_summaries_v0203_tweet_relevance_minicheck*" \) \
  -print | sort

ZIP_PATH="/kaggle/working/reward_scoring_outputs_v2.zip"
!rm -f "$ZIP_PATH"

!zip -r "$ZIP_PATH" \
  data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl \
  data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl \
  reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.csv \
  reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.md \
  reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.csv \
  reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.md \
  reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv \
  reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv \
  reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck \
  reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck

print("\nZip artifact:")
!ls -lh "$ZIP_PATH"

/kaggle/working/role-aware-crisis-summarization
Reward JSONL files:
-rw-r--r-- 1 root root 11M Jun 29 05:59 data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl
-rw-r--r-- 1 root root 11M Jun 29 05:49 data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl

Reward summary tables:
-rw-r--r-- 1 root root  998 Jun 29 05:59 reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.csv
-rw-r--r-- 1 root root  769 Jun 29 05:59 reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.md
-rw-r--r-- 1 root root  904 Jun 29 05:37 reports/tables/t5_baseline_reward_summary_tweet_relevance_minicheck.csv
-rw-r--r-- 1 root root 1006 Jun 29 05:49 reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.csv
-rw-r--r-- 1 root root  769 Jun 29 05:49 reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.md

Presentatio

## Compact T5 vs GPT Comparison Table

In [16]:
import pandas as pd
from pathlib import Path

t5 = pd.read_csv("reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/overall_metrics.csv")
gpt = pd.read_csv("reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck/overall_metrics.csv")

fields = ["reward", "relevance", "factuality", "role_coverage", "urgency", "summary_word_count", "summary_char_length"]
rows = []

for field in fields:
    t5_mean = float(t5.loc[t5["numeric_field"] == field, "mean"].iloc[0])
    gpt_mean = float(gpt.loc[gpt["numeric_field"] == field, "mean"].iloc[0])
    rows.append({
        "metric": field,
        "t5_v2_mean": t5_mean,
        "gpt4o_mean": gpt_mean,
        "gpt_minus_t5": gpt_mean - t5_mean,
    })

comparison = pd.DataFrame(rows)
display(comparison)

comparison_path = Path("reports/tables/t5_v2_vs_gpt4o_reward_comparison.csv")
comparison.to_csv(comparison_path, index=False)
print("Wrote:", comparison_path)

,metric,t5_v2_mean,gpt4o_mean,gpt_minus_t5
0,reward,0.593620,0.617690,0.024071
1,relevance,0.719472,0.724957,0.005485
2,factuality,0.480305,0.478697,-0.001608
3,role_coverage,0.471233,0.508129,0.036897
4,urgency,0.637408,0.713276,0.075868
5,summary_word_count,16.719213,17.388102,0.668889
6,summary_char_length,117.880187,121.824363,3.944176


Wrote: reports/tables/t5_v2_vs_gpt4o_reward_comparison.csv


## Compare Reward Metrics By Exact Role Label

This cell compares T5 v2 and GPT-4o reward metrics by exact role label.

This is useful because the overall mean can hide important differences across:
- single-role summaries
- multi-role summaries
- Police-heavy examples
- EMS and Firefighter examples

In [17]:
import pandas as pd
from pathlib import Path

t5_role = pd.read_csv(
    "reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/by_role_label.csv"
)
gpt_role = pd.read_csv(
    "reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck/by_role_label.csv"
)

score_fields = ["reward", "relevance", "factuality", "role_coverage", "urgency"]

t5_role = t5_role[t5_role["numeric_field"].isin(score_fields)][
    ["group_value", "numeric_field", "count", "mean"]
].rename(columns={"mean": "t5_v2_mean", "count": "t5_count"})

gpt_role = gpt_role[gpt_role["numeric_field"].isin(score_fields)][
    ["group_value", "numeric_field", "count", "mean"]
].rename(columns={"mean": "gpt4o_mean", "count": "gpt_count"})

role_comparison = t5_role.merge(
    gpt_role,
    on=["group_value", "numeric_field"],
    how="inner",
)

role_comparison["gpt_minus_t5"] = role_comparison["gpt4o_mean"] - role_comparison["t5_v2_mean"]

role_comparison = role_comparison.rename(
    columns={
        "group_value": "role",
        "numeric_field": "metric",
    }
)

role_comparison = role_comparison[
    ["role", "metric", "t5_count", "gpt_count", "t5_v2_mean", "gpt4o_mean", "gpt_minus_t5"]
]

output_path = Path("reports/tables/t5_v2_vs_gpt4o_reward_comparison_by_role.csv")
role_comparison.to_csv(output_path, index=False)

display(role_comparison)
print("Wrote:", output_path)

,role,metric,t5_count,gpt_count,t5_v2_mean,gpt4o_mean,gpt_minus_t5
0,EMS,reward,593,593,0.639708,0.729448,0.089740
1,EMS,relevance,593,593,0.757233,0.753576,-0.003657
2,EMS,factuality,593,593,0.480672,0.476748,-0.003924
3,EMS,role_coverage,593,593,0.630551,0.909500,0.278949
4,EMS,urgency,593,593,0.641990,0.823047,0.181057
5,Firefighter,reward,1023,1023,0.669614,0.726688,0.057074
6,Firefighter,relevance,1023,1023,0.749768,0.741738,-0.008030
7,Firefighter,factuality,1023,1023,0.481567,0.481340,-0.000227
8,Firefighter,role_coverage,1023,1023,0.592538,0.865266,0.272727
9,Firefighter,urgency,1023,1023,0.841479,0.868459,0.026979


Wrote: reports/tables/t5_v2_vs_gpt4o_reward_comparison_by_role.csv


## Final Package With Comparison Tables

In [18]:
from pathlib import Path
import zipfile

repo = Path("/kaggle/working/role-aware-crisis-summarization")
zip_path = Path("/kaggle/working/reward_scoring_outputs_v2.zip")

files_and_dirs = [
    repo / "data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl",
    repo / "data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl",
    repo / "reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.csv",
    repo / "reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.md",
    repo / "reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.csv",
    repo / "reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.md",
    repo / "reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv",
    repo / "reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv",
    repo / "reports/tables/t5_v2_vs_gpt4o_reward_comparison.csv",
    repo / "reports/tables/t5_v2_vs_gpt4o_reward_comparison_by_role.csv",
    repo / "reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck",
    repo / "reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck",
]

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in files_and_dirs:
        if item.is_file():
            zf.write(item, item.relative_to(repo))
        elif item.is_dir():
            for file in item.rglob("*"):
                if file.is_file():
                    zf.write(file, file.relative_to(repo))
        else:
            raise FileNotFoundError(item)

print("Zip artifact:")
print(zip_path, f"{zip_path.stat().st_size / (1024 * 1024):.2f} MB")

print("\nCurrent generated artifacts:")
for path in files_and_dirs:
    if path.is_file():
        print(path.relative_to(repo), f"{path.stat().st_size / (1024 * 1024):.2f} MB")
    else:
        print(path.relative_to(repo), "directory")

Zip artifact:
/kaggle/working/reward_scoring_outputs_v2.zip 3.85 MB

Current generated artifacts:
data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl 10.79 MB
data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl 10.28 MB
reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.csv 0.00 MB
reports/tables/t5_baseline_v2_all_predictions_reward_summary_tweet_relevance_minicheck.md 0.00 MB
reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.csv 0.00 MB
reports/tables/gpt4o_initial_summaries_v0203_reward_summary_tweet_relevance_minicheck.md 0.00 MB
reports/tables/t5_baseline_v2_all_predictions_with_rewards_report.csv 4.46 MB
reports/tables/gpt4o_initial_summaries_v0203_with_rewards_report.csv 3.78 MB
reports/tables/t5_v2_vs_gpt4o_reward_comparison.csv 0.00 MB
reports/tables/t5_v2_vs_gpt4o_reward_comparison_by_role.csv 0.00 MB
reports/tables/summary_reward_